# 03 — Sequence models: LSTM and causal Transformer

Both consume the same engineered feature frame the XGBoost model uses, but as variable-length sequences with mask channels. The LSTM is left-to-right by construction; the Transformer encoder is causally masked so attention at hour `t` cannot see hours `> t`.

Compared head-to-head on the same patient split:

- **AUROC** (row-level): expected to be similar to XGBoost.
- **PhysioNet utility**: the metric to actually compare on, because it weighs early prediction.
- **Calibration**: sequence models often underperform here without temperature scaling; flagged for follow-up.

In [ ]:
import sys, os
sys.path.append(os.path.abspath('..'))
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from src.data_loader import load_dataset, patient_train_test_split
from src.features import featurize_many
from src.models.lstm_model import LSTMSepsisModel, LSTMConfig
from src.models.transformer_model import TransformerSepsisModel, TransformerConfig
from src.evaluate import discrimination_metrics, normalised_utility, sweep_threshold_for_utility, threshold_predictions
from src.visualize import plot_risk_trajectory, plot_calibration

In [ ]:
records = load_dataset('../data', subset=5000, seed=0)
train_pool, test = patient_train_test_split(records, test_frac=0.2, seed=0)
train, valid = patient_train_test_split(train_pool, test_frac=0.125, seed=1)
train_frame = featurize_many(train); valid_frame = featurize_many(valid); test_frame = featurize_many(test)
len(train_frame), len(valid_frame), len(test_frame)

## LSTM

In [ ]:
lstm = LSTMSepsisModel(LSTMConfig(hidden_size=128, num_layers=2, epochs=10))
lstm.fit(train_frame, valid_frame=valid_frame)

In [ ]:
v_lab, v_scr = lstm.predict_per_patient(valid_frame)
thr, best = sweep_threshold_for_utility(v_lab, v_scr); print('val utility', best, 'thr', thr)
t_lab, t_scr = lstm.predict_per_patient(test_frame)
lstm_util = normalised_utility(t_lab, threshold_predictions(t_scr, thr))
lstm_disc = discrimination_metrics(t_lab, t_scr)
print('LSTM test:', lstm_util, lstm_disc)

## Transformer (causal)

In [ ]:
tx = TransformerSepsisModel(TransformerConfig(d_model=128, num_layers=3, epochs=10))
tx.fit(train_frame, valid_frame=valid_frame)

In [ ]:
v_lab, v_scr = tx.predict_per_patient(valid_frame)
thr, best = sweep_threshold_for_utility(v_lab, v_scr); print('val utility', best, 'thr', thr)
t_lab, t_scr = tx.predict_per_patient(test_frame)
tx_util = normalised_utility(t_lab, threshold_predictions(t_scr, thr))
tx_disc = discrimination_metrics(t_lab, t_scr)
print('Transformer test:', tx_util, tx_disc)

## Comparison

Side-by-side trajectory plots make the difference between models visually obvious: who alarms earlier, who alarms more falsely on non-septic stays.

In [ ]:
septic = [i for i, l in enumerate(t_lab) if l.any()]
fig, axes = plt.subplots(3, 2, figsize=(13, 8))
for row, idx in enumerate(septic[:3]):
    plot_risk_trajectory(t_lab[idx], lstm.predict_per_patient(test_frame)[1][idx], threshold=thr, ax=axes[row, 0]); axes[row,0].set_title(f'LSTM patient {idx}')
    plot_risk_trajectory(t_lab[idx], t_scr[idx], threshold=thr, ax=axes[row, 1]); axes[row,1].set_title(f'Transformer patient {idx}')
plt.tight_layout(); plt.show()